# 10 — Genie Space (REST API)

## Why this matters for WAF

| WAF pillar | What this notebook proves |
|------------|---------------------------|
| **Interoperability & Usability** | Business users can ask natural-language questions against the *gold* star schema. Genie accuracy is directly a function of table/column comments + tags from notebook 05 — this is where that governance investment pays off. |
| **Data Governance** | Genie respects Unity Catalog permissions and column masks — a user with no access to `attendance` simply cannot query it via Genie either. |
| **Operational Excellence** | Space configuration (tables + instructions) is code — you can diff it, roll it back, and recreate it anywhere. |


In [1]:
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json

load_dotenv()

w = WorkspaceClient()

UC_CATALOG       = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA        = os.getenv("UC_SCHEMA", "mlb_gumbo_waf")
GOLD_SCHEMA      = f"{UC_SCHEMA}_gold"
SQL_WAREHOUSE_ID = os.getenv("SQL_WAREHOUSE_ID")
if not SQL_WAREHOUSE_ID:
    raise ValueError("SQL_WAREHOUSE_ID not set in .env")

G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(G)


alexander_booth.mlb_gumbo_waf_gold


In [2]:
# Tables must be sorted alphabetically for the GenieSpaceExport proto
table_names = sorted([
    f"{G}.dim_batter", f"{G}.dim_date", f"{G}.dim_pitcher", f"{G}.dim_team",
    f"{G}.dim_venue", f"{G}.fact_games", f"{G}.fact_pitches",
    f"{G}.v_pitch_mix_by_type", f"{G}.v_pitcher_leaderboard", f"{G}.v_team_pitching_summary",
])

genie_config = {
    "version": 2,
    "data_sources": {"tables": [{"identifier": t} for t in table_names]},
    "instructions": {
        "text_instructions": [
            {"content": [
                "GUMBO = MLB's granular game + pitch-by-pitch data feed.",
                "A 'pitch event' is a single pitch delivered to the batter. Use fact_pitches for pitch-grain questions.",
                "'Called strike' = call_code = 'C'. 'Strike' in general = is_strike = TRUE. 'In play' = is_in_play = TRUE.",
                "When the user says 'fastball', filter pitch_type_code IN ('FF','FT','FA','SI').",
                "When the user says 'breaking ball', filter pitch_type_code IN ('SL','CU','KC','SV').",
                "Always prefer the gold star schema (dim_* + fact_*) over silver tables for analytics questions.",
                "Velocity is in mph; spin rate is in RPM; plate_x/plate_z are in feet (catcher POV).",
            ]}
        ]
    },
}
serialized = json.dumps(genie_config)
print(f"Config size: {len(serialized):,} chars; tables: {len(table_names)}")


Config size: 1,412 chars; tables: 10


In [3]:
GENIE_TITLE = "MLB GUMBO — Pitch Analyst (WAF)"
current_user = w.current_user.me()
parent_path = f"/Workspace/Users/{current_user.user_name}"

existing_id = None
try:
    resp = w.api_client.do("GET", "/api/2.0/genie/spaces")
    for space in resp.get("spaces", []):
        if space.get("title", "").startswith(GENIE_TITLE):
            existing_id = space["space_id"]
            break
except Exception:
    pass

if existing_id:
    space_id = existing_id
    print(f"Found existing Genie space — reusing: {space_id}")
else:
    response = w.api_client.do(
        "POST", "/api/2.0/genie/spaces",
        body={
            "warehouse_id": SQL_WAREHOUSE_ID,
            "title": GENIE_TITLE,
            "description": "Ask natural-language questions about MLB pitch analytics — pitch mix, velocity, strike rates, and called-strike probabilities across the season.",
            "serialized_space": serialized,
            "parent_path": parent_path,
        },
    )
    space_id = response.get("space_id", response.get("id", "unknown"))
    print(f"Genie space created: {space_id}")

host = os.getenv("DATABRICKS_HOST", "").rstrip("/")
print(f"\nTitle: {GENIE_TITLE}")
print(f"Open:  {host}/genie/rooms/{space_id}")


Found existing Genie space — reusing: 01f13dbc8d261a63a4cd108caf7418e7

Title: MLB GUMBO — Pitch Analyst (WAF)
Open:  https://e2-demo-field-eng.cloud.databricks.com/genie/rooms/01f13dbc8d261a63a4cd108caf7418e7


## Suggested questions to try

1. *Which pitchers throw the hardest fastballs?*
2. *Show me pitch mix by type for the 2025 season.*
3. *Which teams have the highest strike percentage at home?*
4. *How does spin rate correlate with called-strike probability?* (uses `pitch_strike_probability` gold table)
5. *Show me the top 5 games by attendance.* (will redact attendance if the viewer isn't in the `mlb_analytics_team` group — WAF security pillar in action)
